In [2]:
!pip install pypdf



[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\MJYOT\major projects\Rag chatbot
['.env', '.git', '.gitignore', '.idea', '.ipynb_checkpoints', '.venv', 'app1.py', 'novel.pdf', 'rag chatbot .ipynb', 'README.md', 'requirements.txt']


In [4]:
from pypdf import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    pages_data = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if text and text.strip():
            pages_data.append({
                "page_number": page_number,
                "text": text
            })

    return pages_data

In [5]:
file_path = "novel.pdf"
pages = extract_text_from_pdf(file_path)

print(f"Total pages with text: {len(pages)}")
print(pages[0]["text"][:300])

Total pages with text: 6
SSRG International Journal of Humanities and Social Science (SSRG - IJHSS) Volume 4 Issue 5 Sep to Oct 2017 
ISSN: 2394 - 2703                       www.internationaljournalssrg.org                         Page 78 
The Novel: Genres, Concepts Introduction and 
Appreciation 
Uche Nnyagu PhD, Adunchez


In [6]:
!pip install langchain langchain-text-splitters


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pages(pages_data, chunk_size=800, chunk_overlap=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    all_chunks = []
    for page in pages_data:
        chunks = splitter.split_text(page["text"])
        for chunk_text in chunks:
            all_chunks.append({
                "page_number": page["page_number"],
                "text": chunk_text
            })

    return all_chunks

In [21]:
chunks = chunk_pages(pages)

print(f"Total chunks created: {len(chunks)}")
print(chunks[0])

Total chunks created: 45
{'page_number': 1, 'text': 'SSRG International Journal of Humanities and Social Science (SSRG - IJHSS) Volume 4 Issue 5 Sep to Oct 2017 \nISSN: 2394 - 2703                       www.internationaljournalssrg.org                         Page 78 \nThe Novel: Genres, Concepts Introduction and \nAppreciation \nUche Nnyagu PhD, Adunchezor, Ngozi PhD \nDepartment of English, NOCEN \n \nAbstract \nAmong the three genres of literature, prose is \nthe newest and the most popular. Prior to the \neighteenth century, poetry and drama were the only \ngenres of literature. The two had their origin in the \nclassical era. Prose came into existence much later in \nlate eighteenth century and till today; it has remained \nthe most prominent. Prose has as its genre, the novel  \nwhich many people see it as synonyms mainly because'}


In [22]:
!pip install sentence-transformers


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
!pip install --upgrade regex


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(chunks):
    texts = [chunk["text"] for chunk in chunks]
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
embeddings = create_embeddings(chunks)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Length of one embedding vector: {len(embeddings[0])}")
print(embeddings[0][:10])  # preview first 10 numbers of the first chunk's embedding

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Number of embeddings: 45
Length of one embedding vector: 384
[ 0.0234798  -0.03769264 -0.0503774   0.04868235 -0.01675019  0.09133246
  0.00684731  0.01095058  0.05809563  0.07672889]


In [26]:
!pip install faiss-cpu


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import faiss
import numpy as np

def build_faiss_index(embeddings):
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index

In [28]:
index = build_faiss_index(embeddings)

print(f"Number of vectors stored in index: {index.ntotal}")

Number of vectors stored in index: 45


In [29]:
def retrieve_chunks(question, index, chunks, top_k=3):
    question_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(question_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [30]:
question = "What is a novel?"
results = retrieve_chunks(question, index, chunks, top_k=3)

for i, chunk in enumerate(results, start=1):
    print(f"--- Result {i} (Page {chunk['page_number']}) ---")
    print(chunk["text"][:300])
    print()

--- Result 1 (Page 1) ---
Novel is one of the subgenres of literature which 
became prominent in the nineteenth century. The term 
novel denotes a long narrative usually an imaginative 
work of art in form of a prose. According to Kennedy, 
Gioa and Bauerlein (2009), novel is an extended work 
of fictional prose narrative. T

--- Result 2 (Page 1) ---
the most prominent. Prose has as its genre, the novel  
which many people see it as synonyms mainly because 
of its popularity. It is a businessfor the literate unlike 
the other genres that can be performed and appreciated 
by both the literates and the illiterates alike. Despite the 
limitation, n

--- Result 3 (Page 5) ---
which places more than the usual amount of emphasis 
on characterization, and on the motives, circumstances, 
and internal actions which motivate the external action. 
Psychological novel is a realistic fiction not basically 
interested in what happen but in why it happens. It is a 
realistic novel 



In [31]:
!pip install groq


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [39]:
def generate_answer(question, retrieved_chunks):
    context = "\n\n".join([chunk["text"] for chunk in retrieved_chunks])

    prompt = f"""You are a document question-answering assistant.
Answer only from the supplied context. If the answer is not available, say:
"I could not find this information in the uploaded documents."
Do not invent facts.

Context:
{context}

Question: {question}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [40]:
question = "What is a novel?"
retrieved_chunks = retrieve_chunks(question, index, chunks, top_k=3)
answer = generate_answer(question, retrieved_chunks)

print(answer)

According to the documents, a novel is an extended work of fictional prose narrative (Kennedy, Gioa and Bauerlein, 2009) or a type of extended prose fiction that makes use of characters, plot, theme, and setting to pass across its message (Nnyagu, 2015) and nearly always an extended fictional prose narrative (Baldick, 2004).


In [41]:
question = "Who is the president of India?"
retrieved_chunks = retrieve_chunks(question, index, chunks, top_k=3)
answer = generate_answer(question, retrieved_chunks)

print(answer)

I could not find this information in the uploaded documents.


In [37]:
for page in pages:
    if "Final Deliverables" in page["text"] or "deliverables" in page["text"].lower():
        print(f"Found on page {page['page_number']}")
        print(page["text"][:500])
        print("---")

In [1]:
import glob
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()

# ---- 1. Auto-detect the PDF ----
pdf_files = glob.glob("*.pdf")
if not pdf_files:
    raise FileNotFoundError("No PDF found in this folder!")
file_path = pdf_files[0]
print("Using PDF:", file_path)

# ---- 2. Extract text ----
pages = extract_text_from_pdf(file_path)
print(f"Total pages with text extracted: {len(pages)}")
print(f"Last page number found: {pages[-1]['page_number']}")

print("\n--- Preview of last 2 pages (checking the ending) ---")
for p in pages[-2:]:
    print(f"\nPage {p['page_number']}:")
    print(p["text"][:400])

# ---- 3. Chunk, embed, index ----
chunks = chunk_pages(pages)
print(f"\nTotal chunks created: {len(chunks)}")

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

import faiss
import numpy as np
embeddings_np = np.array(embeddings).astype("float32")
index = faiss.IndexFlatL2(embeddings_np.shape[1])
index.add(embeddings_np)
print("FAISS index built with", index.ntotal, "vectors")

# ---- 4. Set up Groq ----
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def retrieve(question, top_k=3):
    q_emb = model.encode([question]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0]]

def ask(question):
    retrieved = retrieve(question)
    context = "\n\n".join(c["text"] for c in retrieved)
    prompt = f"""You are a document question-answering assistant.
Answer only from the supplied context. If the answer is not available, say:
"I could not find this information in the uploaded documents."
Do not invent facts.

Context:
{context}

Question: {question}"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    pages_used = ", ".join(sorted(set(str(c["page_number"]) for c in retrieved)))
    return response.choices[0].message.content, pages_used

# ---- 5. Ask questions in a loop ----
print("\nType a question, or 'quit' to stop.\n")
while True:
    q = input("Your question: ")
    if q.lower() == "quit":
        break
    answer, pages_used = ask(q)
    print(f"\nAnswer: {answer}\nPages used: {pages_used}\n")

Using PDF: novel.pdf


NameError: name 'extract_text_from_pdf' is not defined